# TFMv3 brain en Google Colab con GPU

Notebook reproducible para repetir el experimento de deteccion de anomalias en cerebro (Brain-AD / BraTS2021, modalidad FLAIR) en Google Colab con GPU. La primera parte comprueba el entorno y ejecuta una prueba reducida. La segunda parte lanza la campana cientifica del autoencoder simple (AE) a 256x256 con el score hibrido (senales globales + MAE, peso calibrado en validacion) con la semilla 42 y compara el AUROC con la referencia local: **0.9080**.

Antes de ejecutar, selecciona `Runtime > Change runtime type > GPU`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import time

REPO_URL = "https://github.com/alerodriargui/TFMv3.git"
BRANCH = "brain"
PROJECT_DIR = Path("/content/TFMv3")
IMAGE_SIZE = 256
MODEL_NAME = "ae"
BASELINE_AUROC_LOCAL = 0.9080
MOUNT_DRIVE = True
BACKUP_DIR = Path("/content/drive/MyDrive/TFMv3_colab_backup")

print("Python", sys.version)
print("Platform", platform.platform())

## 1. Clonar el repositorio e instalar dependencias

Colab ya incluye PyTorch compatible con su runtime CUDA. Por eso se instalan solo las dependencias de apoyo desde `requirements-colab.txt` y despues se comprueba la version real de PyTorch/CUDA.

In [ ]:
if PROJECT_DIR.exists():
    print(f"Repositorio presente; sincronizando {BRANCH}...")
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Trabajo en", Path.cwd())
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
# Verificacion: rama correcta y codigo del score hibrido
_branch = subprocess.check_output([
    "git", "rev-parse", "--abbrev-ref", "HEAD"
], text=True).strip()
_src = Path("tfm_ae/experiment.py").read_text(encoding="utf-8")
_feat = Path("tfm_ae/features.py").is_file()
_train = Path("tfm_ae/train.py").read_text(encoding="utf-8")
print("Rama:", _branch)
print("features.py presente:", _feat)
print("experiment.py contiene calibrate_hybrid_scores:", "calibrate_hybrid_scores" in _src)
print("train.py soporta --score-mode:", "score_mode" in _train)
assert _branch == BRANCH, f"Rama incorrecta: {_branch} (esperada {BRANCH})"
assert _feat, "Falta tfm_ae/features.py (senales globales)"
assert "calibrate_hybrid_scores" in _src, "Falta el score hibrido en experiment.py"
assert "score_mode" in _train, "Falta --score-mode en train.py"


In [ ]:
%pip install -q -r requirements-colab.txt

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive montado:", Path("/content/drive/MyDrive").is_dir())
else:
    print("MOUNT_DRIVE = False: sin respaldo en Drive")


## 2. Comprobar GPU, PyTorch y CUDA

Esta celda falla de forma explicita si la sesion no tiene CUDA. Si ocurre, cambia el tipo de runtime a GPU y vuelve a ejecutar desde el principio.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponible:", cuda_available)
print("CUDA PyTorch:", torch.version.cuda)

if cuda_available:
    print("GPU:", torch.cuda.get_device_name(0))
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi no esta disponible en esta runtime")
else:
    raise RuntimeError("Esta ejecucion necesita GPU. Activa Runtime > Change runtime type > GPU.")

## 3. Descargar Brain-AD (BraTS2021)

El zip `Brain_AD.zip` (aprox. 382 MB) se extrae en el entorno de Colab (`/content`), nunca dentro de Drive. Por defecto se descarga desde el Google Drive de BMAD la primera vez. La celda detecta automaticamente la carpeta raiz que contiene `train/`, `valid/` y `test/` tras la extraccion.

**Opcion recomendada (cache persistente):** sube `Brain_AD.zip` una sola vez a Drive (por ejemplo a `MyDrive/datasets/Brain_AD.zip`), pon `USE_DRIVE_ZIP = True` y monta Drive. Las siguientes sesiones extraeran desde ese zip sin volver a descargar 382 MB.

In [ ]:
import gdown
import zipfile

DATA_DIR = Path("/content/data")
ZIP_PATH = DATA_DIR / "Brain_AD.zip"
ZIP_URL = "https://drive.google.com/uc?id=1xuapqiL1s18eMeKMd-Vi1Cqp2wp7Pghp"

USE_DRIVE_ZIP = False
DRIVE_ZIP = Path("/content/drive/MyDrive/datasets/Brain_AD.zip")
DATA_DIR.mkdir(parents=True, exist_ok=True)

def find_dataset_root():
    for candidate in Path("/content").iterdir():
        if (
            candidate.is_dir()
            and (candidate / "train" / "good").is_dir()
            and (candidate / "test").is_dir()
        ):
            return candidate
    return None

if not ZIP_PATH.is_file() and not (USE_DRIVE_ZIP and DRIVE_ZIP.is_file()):
    print("Descargando Brain_AD.zip (aprox. 382 MB)...")
    gdown.download(ZIP_URL, str(ZIP_PATH), quiet=False)
    if not zipfile.is_zipfile(ZIP_PATH):
        ZIP_PATH.unlink()
        raise RuntimeError("Descarga corrupta. Vuelve a ejecutar esta celda.")

source_zip = (
    DRIVE_ZIP
    if USE_DRIVE_ZIP and DRIVE_ZIP.is_file() and zipfile.is_zipfile(DRIVE_ZIP)
    else ZIP_PATH
)

DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    print("Extrayendo en /content...")
    with zipfile.ZipFile(source_zip) as archive:
        archive.extractall(Path("/content"))
    DATA_ROOT = find_dataset_root()
    if DATA_ROOT is None:
        raise RuntimeError("No se encontro la estructura train/valid/test tras extraer el zip")

print("Dataset detectado en", DATA_ROOT)
os.environ["TFM_DATA_ROOT"] = str(DATA_ROOT)
print("TFM_DATA_ROOT=", os.environ["TFM_DATA_ROOT"])

In [ ]:
from tfm_ae.data import find_images, resolve_data_root, split_dir

root = resolve_data_root(DATA_ROOT)
expected = {
    "train/good": split_dir(root, "train") / "good",
    "valid/good": split_dir(root, "val") / "good",
    "valid/Ungood": split_dir(root, "val") / "Ungood",
    "test/good": split_dir(root, "test") / "good",
    "test/Ungood": split_dir(root, "test") / "Ungood",
}

dataset_counts = {}
for name, folder in expected.items():
    if not folder.is_dir():
        raise FileNotFoundError(f"Falta la carpeta {folder}")
    dataset_counts[name] = len(find_images(folder))

print("Dataset:", root)
print(json.dumps(dataset_counts, indent=2, ensure_ascii=False))

## 4. Prueba reducida en GPU

Esta prueba usa pocas imagenes para confirmar que el pipeline completo entrena, calibra, evalua y escribe artefactos sobre GPU (score hibrido incluido) antes de lanzar la campana completa.

In [ ]:
SMOKE_OUTPUT = Path("results/colab_smoke")
smoke_command = [
    sys.executable,
    "-m",
    "tfm_ae.train",
    "--seeds",
    "13",
    "--epochs",
    "1",
    "--model",
    str(MODEL_NAME),
    "--image-size",
    str(IMAGE_SIZE),
    "--batch-size",
    "32",
    "--max-train-images",
    "128",
    "--max-eval-images-per-class",
    "32",
    "--score-mode",
    "hybrid",
    "--output-root",
    str(SMOKE_OUTPUT),
    "--data-root",
    str(root),
]

print("Comando:", " ".join(smoke_command))
import shutil
shutil.rmtree(SMOKE_OUTPUT, ignore_errors=True)
started = time.perf_counter()
subprocess.run(smoke_command, check=True)
print(f"Duracion prueba reducida: {time.perf_counter() - started:.1f} s")

In [ ]:
smoke_metrics_path = SMOKE_OUTPUT / "ae_seed13" / "metrics.json"
smoke_metrics = json.loads(smoke_metrics_path.read_text(encoding="utf-8"))
print(json.dumps({
    "device": smoke_metrics["device"],
    "elapsed_seconds": smoke_metrics["elapsed_seconds"],
    "test_auroc": smoke_metrics["test"]["auroc"],
    "test_balanced_accuracy": smoke_metrics["test"]["balanced_accuracy"],
    "weight": smoke_metrics["calibration"]["weight"],
    "scientific_run": smoke_metrics["scientific_run"],
}, indent=2))

assert smoke_metrics["device"] == "cuda", smoke_metrics["device"]

## 5. Campana completa del AE final (AE 256x256, score hibrido)

Ejecuta el AE con la semilla 42, 3 epocas, batch 32 y test completo. Esta es la ejecucion comparable con la referencia local: **AUROC 0.9080**.

**Respaldo en Drive:** tras cada semilla, la celda copia sus resultados y el checkpoint a `MyDrive/TFMv3_colab_backup/<fecha>/`. Si la sesion se desconecta, recupera esa carpeta desde Drive sin volver a entrenar.

In [ ]:
FULL_OUTPUT = Path("results/colab_ae_full")
SEEDS = [42]
BASE_COMMAND = [
    sys.executable,
    "-m",
    "tfm_ae.train",
    "--epochs",
    "3",
    "--model",
    str(MODEL_NAME),
    "--image-size",
    str(IMAGE_SIZE),
    "--batch-size",
    "32",
    "--score-mode",
    "hybrid",
    "--output-root",
    str(FULL_OUTPUT),
    "--data-root",
    str(root),
]

import shutil
from datetime import datetime

backup_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = BACKUP_DIR / backup_timestamp

def backup_progress():
    if not MOUNT_DRIVE:
        return
    backup_dir.mkdir(parents=True, exist_ok=True)
    for seed_dir in sorted(FULL_OUTPUT.glob("ae_seed*")):
        target = backup_dir / "results_colab_ae_full" / seed_dir.name
        shutil.copytree(seed_dir, target, dirs_exist_ok=True)
    for rel in [Path("checkpoints/modelo_autoencoder.pt"), Path("results/resultados.csv")]:
        if rel.is_file():
            shutil.copy2(rel, backup_dir / rel.name)
    print("Respaldo en Drive:", backup_dir)

shutil.rmtree(FULL_OUTPUT, ignore_errors=True)
started = time.perf_counter()
for seed in SEEDS:
    subprocess.run(BASE_COMMAND + ["--seeds", str(seed)], check=True)
    backup_progress()
full_elapsed = time.perf_counter() - started
print(f"Duracion campana AE completa: {full_elapsed:.1f} s")

In [ ]:
import pandas as pd

rows = []
for metrics_path in sorted(FULL_OUTPUT.glob("ae_seed*/metrics.json")):
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append({
        "seed": metrics["config"]["seed"],
        "device": metrics["device"],
        "selected_epoch": metrics["selected_epoch"],
        "elapsed_seconds": metrics["elapsed_seconds"],
        "auroc": metrics["test"]["auroc"],
        "balanced_accuracy": metrics["test"]["balanced_accuracy"],
        "weight": metrics["calibration"]["weight"],
        "scientific_run": metrics["scientific_run"],
    })

summary = pd.DataFrame(rows).sort_values("seed")
display(summary)

mean_auroc = float(summary["auroc"].mean())
print(f"AUROC medio Colab GPU {IMAGE_SIZE}x{IMAGE_SIZE}: {mean_auroc:.4f}")
print(f"AUROC referencia local: {BASELINE_AUROC_LOCAL:.4f}")
print(f"Diferencia: {mean_auroc - BASELINE_AUROC_LOCAL:+.4f}")

assert set(summary["seed"]) == set(SEEDS)
assert (summary["device"] == "cuda").all()
assert summary["scientific_run"].all()

## 6. Guardar artefactos

Se guardan resultados, metricas, tiempos, reconstrucciones y checkpoints en `/content/TFMv3_colab_outputs`. Si ademas tienes Google Drive montado, se copian tambien a Drive.

In [ ]:
from datetime import datetime
import shutil

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
local_output = Path("/content/TFMv3_colab_outputs") / timestamp
local_output.mkdir(parents=True, exist_ok=True)

for path in [Path("results/resultados.csv"), Path("checkpoints/modelo_autoencoder.pt")]:
    if path.exists():
        shutil.copy2(path, local_output / path.name)

shutil.copytree(FULL_OUTPUT, local_output / "results_colab_ae_full", dirs_exist_ok=True)
shutil.copytree(SMOKE_OUTPUT, local_output / "results_colab_smoke", dirs_exist_ok=True)

run_manifest = {
    "repo_url": REPO_URL,
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "data_root": str(root),
    "dataset_counts": dataset_counts,
    "smoke_command": smoke_command,
    "full_command": BASE_COMMAND + ["--seeds"] + [str(seed) for seed in SEEDS],
    "full_elapsed_seconds": full_elapsed,
    "image_size": IMAGE_SIZE,
    "model_name": MODEL_NAME,
    "score_mode": "hybrid",
    "baseline_auroc_local": BASELINE_AUROC_LOCAL,
    "colab_auroc_mean": mean_auroc,
    "colab_minus_baseline_auroc": mean_auroc - BASELINE_AUROC_LOCAL,
}
(local_output / "run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Artefactos guardados en", local_output)

if Path("/content/drive").exists():
    drive_output = Path("/content/drive/MyDrive/TFMv3_colab_outputs") / timestamp
    drive_output.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output, drive_output, dirs_exist_ok=True)
    print("Copia adicional en", drive_output)

## Interpretacion de diferencias CPU local vs GPU Colab

Si los AUROC difieren ligeramente de la referencia local (0.9080), revisa primero el manifiesto: GPU asignada, versiones de PyTorch/CUDA, conteos del dataset y commit. El entrenamiento fija semillas y configura cuDNN en modo determinista, pero pequenas diferencias numericas pueden aparecer entre CPU y GPU o entre versiones de PyTorch/CUDA. Una diferencia relevante debe documentarse junto con esos datos y con los `metrics.json` de cada semilla.

## 7. Descargar artefactos de la ejecucion

Empaqueta en un zip los resultados, metricas, reconstrucciones y checkpoint, y lo descarga a tu equipo.

In [ ]:
import zipfile
from datetime import datetime
from pathlib import Path

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
archive = Path(f"/content/TFMv3_colab_brain_{stamp}.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    candidates = [
        Path("/content/TFMv3_colab_outputs"),
        Path("results/colab_ae_full"),
        Path("results/colab_smoke"),
        Path("results/resultados.csv"),
        Path("checkpoints/modelo_autoencoder.pt"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            zf.write(candidate, candidate.name)
        elif candidate.is_dir():
            for path in candidate.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(candidate.parent))

print("Zip creado:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
from google.colab import files
files.download(str(archive))